# Spec 6 Extended — Directional Exit (Upward vs Downward)

Decomposing Exit into **Upward** (MIT escape) vs **Downward** (relapse).

**How to use this notebook in Google Colab**
1. Run the cells top to bottom (Shift+Enter, or *Runtime → Run all*).
2. When the upload cell prompts you, pick your `final_panel_F_rebuilt.csv`.
3. The analysis runs automatically and writes `exit_directional_ppp.csv`,
   which you can download from the last cell.


## 0. Install / check packages
(Colab already has these, but this makes sure.)

In [ ]:
import numpy as np, pandas as pd
import statsmodels.api as sm
from scipy import optimize, stats

## 1. Upload your panel file
Run this cell, then choose `final_panel_F_rebuilt.csv` from your computer.

In [ ]:
from google.colab import files
uploaded = files.upload()
# Use whatever filename you uploaded (handles renamed files too)
FILENAME = list(uploaded.keys())[0]
print("Using file:", FILENAME)

Saving final_panel_F_rebuilt.csv to final_panel_F_rebuilt.csv
Using file: final_panel_F_rebuilt.csv


In [ ]:
raw = pd.read_csv(FILENAME)
print(f"Raw rows: {len(raw):,}")
df = (raw.drop_duplicates(subset=["iso3", "year"])
         .sort_values(["iso3", "year"])
         .reset_index(drop=True))
print(f"After dedup: {len(df):,} ({df['iso3'].nunique()} countries, "
      f"{df['year'].min()}-{df['year'].max()})")

Raw rows: 10,424
After dedup: 3,255 (217 countries, 2010-2024)


In [ ]:
"""
Spec 6 — CORRECTED directional-exit analysis.
Fixes carried out:
  (1) dedup the panel (the upstream merge triple/11x-replicated rows)
  (2) build time-varying income bands from gdp_pc_ppp -> ExitUp / ExitDown
  (3) rebuild the Vuln indicator two ways and compare against the old one:
        - Vuln_old   : the algebraic-threshold indicator already in the file
        - Vuln_AND   : strict logical AND (high rho, low H, low KIS, low I)
        - Vuln_net   : rho net of absorption, rho*(1-H)*(1-KIS) above 75th pctile
  (4) re-run the parsimonious probits for each Vuln definition
  (5) Firth-penalized logit for the pooled rare-events outcome
  (6) split G x sigma_kis by income group (the wrong-signed eta6 diagnostic)
"""
import numpy as np, pandas as pd
import statsmodels.api as sm
from scipy import optimize, stats
pd.set_option("display.width", 160)



# ----------------------------------------------------------------------
# 2. time-varying income bands -> directional exits
# ----------------------------------------------------------------------
PPP_LMI, PPP_UMI, PPP_HI = 4_000, 11_750, 30_000
def band(x):
    if pd.isna(x): return np.nan
    if x < PPP_LMI: return "L"
    if x < PPP_UMI: return "LM"
    if x < PPP_HI:  return "UM"
    return "H"
df["ig"]      = df["gdp_pc_ppp"].apply(band)
df["ig_lag"]  = df.groupby("iso3")["ig"].shift(1)
df["was_MI"]  = df["ig_lag"].isin(["LM","UM"]).astype(int)
df["ExitUp"]  = ((df["was_MI"]==1) & (df["ig"]=="H")).astype(int)
df["ExitDown"]= ((df["was_MI"]==1) & (df["ig"]=="L")).astype(int)
df["ExitAny"] = df["ExitUp"] + df["ExitDown"]

# ----------------------------------------------------------------------
# 3. rebuild Vuln
#    Use the same structural primitives the theory's threshold uses.
#    Thresholds computed within the MI risk set so they mean "relative to peers".
# ----------------------------------------------------------------------
risk = df[df["was_MI"]==1].copy()

med = {c: risk[c].median() for c in ["rho","H","sigma_kis","I"]}
risk["Vuln_old"] = risk["Vuln"].fillna(0).astype(int)          # what was in the file
risk["Vuln_AND"] = ((risk["rho"]       > med["rho"]) &
                    (risk["H"]         < med["H"]) &
                    (risk["sigma_kis"] < med["sigma_kis"]) &
                    (risk["I"]         < med["I"])).astype(int)
risk["rho_net"]  = risk["rho"]*(1-risk["H"])*(1-risk["sigma_kis"])
cut              = risk["rho_net"].quantile(0.75)
risk["Vuln_net"] = (risk["rho_net"] > cut).astype(int)

print("\nVuln set sizes within the MI risk set (N={}):".format(len(risk)))
for v in ["Vuln_old","Vuln_AND","Vuln_net"]:
    print(f"  {v:10s}: flagged = {int(risk[v].sum()):4d}  "
          f"({risk[v].mean()*100:4.1f}% of risk set)")

# ----------------------------------------------------------------------
# 4. who gets flagged? show the composition for each definition
# ----------------------------------------------------------------------
traits = ["H","sigma_kis","rho","I","gini","G","F","gdp_pc_ppp"]
traits = [t for t in traits if t in risk.columns]
print("\n" + "="*72)
print("COMPOSITION CHECK — mean structural traits of flagged vs not")
print("="*72)
for v in ["Vuln_old","Vuln_AND","Vuln_net"]:
    print(f"\n--- {v} ---")
    print(risk.groupby(v)[traits].mean().round(3).T)

# show which countries the strict definition concentrates on
print("\nTop countries by share of years flagged Vuln_AND:")
share = (risk.groupby("iso3")["Vuln_AND"].mean()
              .sort_values(ascending=False))
print(share[share>0].head(20).round(2).to_string())

# ----------------------------------------------------------------------
# 5. parsimonious probits for each Vuln definition
# ----------------------------------------------------------------------
def run_probit(data, y, X):
    sub = data.dropna(subset=X+[y]).copy()
    yy  = sub[y].astype(int)
    if yy.sum() < 3:
        return None, len(sub), int(yy.sum())
    XX  = sm.add_constant(sub[X])
    try:
        m = sm.Probit(yy, XX).fit(disp=False, maxiter=400)
        return m, len(sub), int(yy.sum())
    except Exception as e:
        return str(e), len(sub), int(yy.sum())

print("\n" + "="*72)
print("PARSIMONIOUS PROBITS — outcome = ExitUp (MI->HI), 3 Vuln definitions")
print("="*72)
for v in ["Vuln_old","Vuln_AND","Vuln_net"]:
    X = ["F_l","G_l","FG_l",v]
    m,N,ev = run_probit(risk,"ExitUp",X)
    print(f"\nExitUp ~ F_l + G_l + FG_l + {v}   (N={N}, events={ev})")
    if hasattr(m,"params"):
        out=pd.DataFrame({"coef":m.params,"se":m.bse,"z":m.tvalues,"p":m.pvalues}).round(4)
        print(out)
    else:
        print("  ", m)

# ----------------------------------------------------------------------
# 6. Firth-penalized logit (manual) for pooled ExitAny — rare events
# ----------------------------------------------------------------------
def firth_logit(X, y, max_iter=200, tol=1e-6):
    """Firth's penalized MLE. X includes intercept column. Returns coef, se, p."""
    X = np.asarray(X, float); y = np.asarray(y, float)
    n,p = X.shape
    beta = np.zeros(p)
    for _ in range(max_iter):
        eta = X@beta
        pr  = 1/(1+np.exp(-eta))
        W   = pr*(1-pr)
        XtWX = X.T*(W) @ X
        try: XtWX_inv = np.linalg.inv(XtWX)
        except np.linalg.LinAlgError: XtWX_inv = np.linalg.pinv(XtWX)
        # hat diagonal
        Wsqrt = np.sqrt(W)
        Xs = X*Wsqrt[:,None]
        H  = Xs @ XtWX_inv @ Xs.T
        h  = np.diag(H)
        # Firth-modified score
        U = X.T @ (y - pr + h*(0.5 - pr))
        delta = XtWX_inv @ U
        beta_new = beta + delta
        if np.max(np.abs(beta_new-beta)) < tol:
            beta = beta_new; break
        beta = beta_new
    eta = X@beta; pr = 1/(1+np.exp(-eta)); W = pr*(1-pr)
    cov = np.linalg.pinv(X.T*(W)@X)
    se  = np.sqrt(np.diag(cov))
    z   = beta/se
    pval= 2*(1-stats.norm.cdf(np.abs(z)))
    return beta, se, z, pval

print("\n" + "="*72)
print("FIRTH-PENALIZED LOGIT — pooled ExitAny (rare-events robustness)")
print("="*72)
for v in ["Vuln_old","Vuln_AND","Vuln_net"]:
    cols = ["F_l","G_l","FG_l",v]
    sub  = risk.dropna(subset=cols+["ExitAny"]).copy()
    X    = sm.add_constant(sub[cols]).values
    y    = sub["ExitAny"].values
    b,se,z,pv = firth_logit(X,y)
    names = ["const"]+cols
    print(f"\nExitAny (Firth) ~ F_l+G_l+FG_l+{v}  (N={len(sub)}, events={int(y.sum())})")
    print(pd.DataFrame({"coef":b,"se":se,"z":z,"p":pv}, index=names).round(4))

# ----------------------------------------------------------------------
# 7. G x sigma_kis split by income group  (the eta6 wrong-sign diagnostic)
#    Regress growth on G, sigma_kis, G*sigma_kis within UMI vs LMI.
# ----------------------------------------------------------------------
print("\n" + "="*72)
print("eta6 DIAGNOSTIC — growth ~ G + sigma_kis + G*sigma_kis, by income band")
print("="*72)
work = df.dropna(subset=["gdp_pc_growth","G","sigma_kis","gdp_pc_ppp"]).copy()
work["Gxsk"] = work["G"]*work["sigma_kis"]
work["band"] = work["gdp_pc_ppp"].apply(band)
for grp,label in [(["LM"],"Lower-middle (LM)"),
                  (["UM"],"Upper-middle (UM)"),
                  (["LM","UM"],"All middle-income")]:
    s = work[work["band"].isin(grp)]
    if len(s) < 30:
        print(f"\n{label}: N={len(s)} too small"); continue
    X = sm.add_constant(s[["G","sigma_kis","Gxsk"]])
    m = sm.OLS(s["gdp_pc_growth"], X).fit(cov_type="HC1")
    print(f"\n{label}  (N={len(s)})")
    print(pd.DataFrame({"coef":m.params,"se":m.bse,"p":m.pvalues}).round(4))

# ----------------------------------------------------------------------
# 8. save the rebuilt risk set for downstream use
# ----------------------------------------------------------------------
# keep = ["iso3","year","gdp_pc_ppp","ig","ig_lag","was_MI",
#         "ExitUp","ExitDown","ExitAny",
#         "Vuln_old","Vuln_AND","Vuln_net","rho_net",
#         "F_l","G_l","FG_l","H","sigma_kis","rho","I","gini","G","F"]
# keep=[c for c in keep if c in risk.columns]
# risk[keep].to_csv("/home/claude/exit_directional_vuln_rebuilt.csv", index=False)
# print("\nsaved exit_directional_vuln_rebuilt.csv  rows", len(risk))


Vuln set sizes within the MI risk set (N=1433):
  Vuln_old  : flagged =   74  ( 5.2% of risk set)
  Vuln_AND  : flagged =    0  ( 0.0% of risk set)
  Vuln_net  : flagged =  186  (13.0% of risk set)

COMPOSITION CHECK — mean structural traits of flagged vs not

--- Vuln_old ---
Vuln_old            0          1
H               0.329      0.426
sigma_kis       0.040      0.097
rho             0.067      0.160
I              -0.302      0.417
gini            0.401      0.402
G              -0.233      0.134
F              -0.501     -0.059
gdp_pc_ppp  13587.255  25333.100

--- Vuln_AND ---
Vuln_AND            0
H               0.336
sigma_kis       0.045
rho             0.071
I              -0.264
gini            0.401
G              -0.210
F              -0.476
gdp_pc_ppp  14196.360

--- Vuln_net ---
Vuln_net            0          1
H               0.332      0.355
sigma_kis       0.043      0.053
rho             0.064      0.120
I              -0.318      0.088
gini            0.403    

/tmp/ipykernel_1230/523375702.py:136: RuntimeWarning: invalid value encountered in divide
  z   = beta/se



ExitAny (Firth) ~ F_l+G_l+FG_l+Vuln_AND  (N=1084, events=18)
            coef      se        z       p
const    -4.1600  0.3640 -11.4275  0.0000
F_l       0.1318  0.4058   0.3249  0.7453
G_l       1.6860  0.4420   3.8147  0.0001
FG_l      0.3765  0.5293   0.7113  0.4769
Vuln_AND  0.0000  0.0000      NaN     NaN

ExitAny (Firth) ~ F_l+G_l+FG_l+Vuln_net  (N=1084, events=18)
            coef      se       z       p
const    -5.9643  0.7188 -8.2974  0.0000
F_l      -0.3992  0.4570 -0.8735  0.3824
G_l       2.1157  0.5801  3.6468  0.0003
FG_l      1.3867  0.5527  2.5090  0.0121
Vuln_net  2.8837  0.5979  4.8233  0.0000

eta6 DIAGNOSTIC — growth ~ G + sigma_kis + G*sigma_kis, by income band

Lower-middle (LM)  (N=375)
              coef       se       p
const       4.8865   0.6111  0.0000
G           3.2156   0.8453  0.0001
sigma_kis -64.4431  21.4785  0.0027
Gxsk      -56.4748  29.7756  0.0579

Upper-middle (UM)  (N=529)
             coef       se       p
const      2.4627   0.3835  0.0000


In [ ]:
"""
eta6 DIAGNOSTIC v2 — Testing whether the negative G*sigma_KIS coefficient in
LMICs is driven by premature deindustrialization (Rodrik 2016).

Logic:
  Original finding: in LMICs, G*sigma_KIS = -56.5 (p=0.058)
                    in UMICs, G*sigma_KIS = -1.7  (p=0.89)
  The sign and magnitude shift with income — suggests the negative interaction
  is income-mediated, not a uniform AI-vs-KIS effect.

Strategy: instead of binning by income group, fit the income mediation
explicitly with a smooth quadratic in log(GDP/cap). If the negative
G*sigma_KIS coefficient is fully captured by the (G*sigma_KIS)*log(y)
and (G*sigma_KIS)*log(y)^2 terms — and the residual G*sigma_KIS goes
to zero or flips positive — premature deindustrialization is confirmed
as the confound.

Outputs:
  Spec A: baseline (replicates the LMIC result)
  Spec B: + linear income mediation of G*sigma_KIS
  Spec C: + quadratic income mediation (the headline test)
  Spec D: marginal effect of G*sigma_KIS at income deciles
  Spec E: same on UMIC sub-sample as a placebo
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import warnings; warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# Load and clean
# -----------------------------------------------------------------------------
df = (pd.read_csv("final_panel_F_rebuilt.csv")
        .drop_duplicates(subset=["iso3","year"])
        .sort_values(["iso3","year"])
        .reset_index(drop=True))

# Compute log GDP per capita PPP, lagged
df["log_gdp"]   = np.log(df["gdp_pc_ppp"])
df["log_gdp_l"] = df.groupby("iso3")["log_gdp"].shift(1)
df["log_gdp_l_sq"] = df["log_gdp_l"]**2

# Construct interactions
df["G_sk_l"]            = df["G_l"] * df["sigma_kis"]                # base interaction
df["G_sk_l_x_logy"]     = df["G_sk_l"] * df["log_gdp_l"]             # linear mediation
df["G_sk_l_x_logy_sq"]  = df["G_sk_l"] * df["log_gdp_l_sq"]          # quadratic mediation
df["sigma_kis_x_logy"]  = df["sigma_kis"] * df["log_gdp_l"]          # control: KIS-income
df["sigma_kis_x_logy_sq"] = df["sigma_kis"] * df["log_gdp_l_sq"]

# Sample: middle-income only (matches the LMIC alarm bell)
mi_sub = df[df["mi_status"]==1].copy()
lm_sub = df[df["income_group"]=="LM"].copy()
um_sub = df[df["income_group"]=="UM"].copy()

# Standard controls (lagged)
controls = ["log_gdp_l", "trade_pct_gdp_l", "pop_growth_l",
            "inflation_l", "investment_pct_gdp_l", "credit_to_private_pct_gdp_l"]
controls = [c for c in controls if c in df.columns]

def fit_panel(data, regressors, outcome="g_growth_l", entity="iso3"):
    """OLS with two-way fixed effects via dummies (simple, no extras)."""
    needed = regressors + [outcome, entity, "year"]
    sub = data.dropna(subset=needed).copy()
    if len(sub) < 50:
        return None, sub
    # Two-way FE via demeaning is cleaner; use within transformation
    for col in regressors + [outcome]:
        sub[col] = (sub[col] - sub.groupby(entity)[col].transform("mean")
                              - sub.groupby("year")[col].transform("mean")
                              + sub[col].mean())
    y = sub[outcome]
    X = sm.add_constant(sub[regressors])
    m = sm.OLS(y, X).fit(cov_type="cluster",
                          cov_kwds={"groups": sub[entity]})
    return m, sub

def print_result(name, m, vars_of_interest):
    print(f"\n{name}  (N={int(m.nobs)})")
    print("-"*72)
    out = pd.DataFrame({"coef": m.params, "se": m.bse,
                        "t": m.tvalues, "p": m.pvalues})
    print(out.loc[[v for v in vars_of_interest if v in m.params.index]].round(4))

# -----------------------------------------------------------------------------
# Identify the right growth variable
# -----------------------------------------------------------------------------
growth_candidates = [c for c in df.columns if "growth" in c.lower() and
                     "pop" not in c.lower()][:5]
print(f"Growth-like columns found: {growth_candidates}")
# Pick most populated
growth_var = max(growth_candidates,
                 key=lambda c: df[c].notna().sum()) if growth_candidates else None
print(f"Using growth variable: {growth_var}")

# Need contemporaneous outcome
if growth_var:
    df["g_outcome"] = df[growth_var]
    lm_sub["g_outcome"] = df.loc[lm_sub.index, "g_outcome"]
    um_sub["g_outcome"] = df.loc[um_sub.index, "g_outcome"]
    mi_sub["g_outcome"] = df.loc[mi_sub.index, "g_outcome"]

# -----------------------------------------------------------------------------
# SPEC A — Replicate the LMIC baseline
# -----------------------------------------------------------------------------
print("\n" + "="*72)
print("SPEC A — Baseline (replicate LMIC result)")
print("="*72)
regs_A = ["G_l", "sigma_kis", "G_sk_l"] + controls
mA, _ = fit_panel(lm_sub, regs_A, outcome="g_outcome")
if mA is not None:
    print_result("LMIC sub-sample, baseline", mA, regs_A)

# -----------------------------------------------------------------------------
# SPEC B — Add LINEAR income mediation of G*sigma_KIS
# -----------------------------------------------------------------------------
print("\n" + "="*72)
print("SPEC B — + linear income mediation of G*sigma_KIS")
print("="*72)
regs_B = ["G_l", "sigma_kis", "G_sk_l", "G_sk_l_x_logy",
          "sigma_kis_x_logy"] + controls
mB_lm, _ = fit_panel(lm_sub, regs_B, outcome="g_outcome")
if mB_lm is not None:
    print_result("LMIC sub-sample, linear mediation", mB_lm,
                 ["G_l","sigma_kis","G_sk_l","G_sk_l_x_logy"])

# -----------------------------------------------------------------------------
# SPEC C — HEADLINE: quadratic income mediation
# -----------------------------------------------------------------------------
print("\n" + "="*72)
print("SPEC C — HEADLINE: full income-mediated G*sigma_KIS")
print("="*72)
regs_C = ["G_l", "sigma_kis", "G_sk_l",
          "G_sk_l_x_logy", "G_sk_l_x_logy_sq",
          "sigma_kis_x_logy", "sigma_kis_x_logy_sq",
          "log_gdp_l_sq"] + controls

# Run on the full MI sample so we span the full income range
mC, sub_C = fit_panel(mi_sub, regs_C, outcome="g_outcome")
if mC is not None:
    print_result("Full MI sample, quadratic mediation", mC,
                 ["G_l","sigma_kis","G_sk_l","G_sk_l_x_logy",
                  "G_sk_l_x_logy_sq"])

# -----------------------------------------------------------------------------
# SPEC D — Marginal effect of G*sigma_KIS at income deciles
# -----------------------------------------------------------------------------
if mC is not None:
    print("\n" + "="*72)
    print("SPEC D — Implied marginal effect of G*sigma_KIS along income")
    print("="*72)
    b1 = mC.params.get("G_sk_l", 0)
    b2 = mC.params.get("G_sk_l_x_logy", 0)
    b3 = mC.params.get("G_sk_l_x_logy_sq", 0)
    v  = mC.cov_params().loc[
        ["G_sk_l","G_sk_l_x_logy","G_sk_l_x_logy_sq"],
        ["G_sk_l","G_sk_l_x_logy","G_sk_l_x_logy_sq"]
    ].values

    print(f"\nFitted: dg/d(G*sk) = {b1:+.2f} + {b2:+.2f}*log(y) + {b3:+.2f}*log(y)^2")

    # Implied turning point (where dg/d(Gsk) = 0 has roots)
    if abs(b3) > 1e-10:
        disc = b2**2 - 4*b3*b1
        if disc >= 0:
            r1 = (-b2 - np.sqrt(disc))/(2*b3)
            r2 = (-b2 + np.sqrt(disc))/(2*b3)
            print(f"Roots (log y where effect = 0): {r1:.2f}, {r2:.2f}  "
                  f"=> y ≈ ${np.exp(r1):,.0f} and ${np.exp(r2):,.0f} PPP")

    # Tabulate at income deciles
    deciles = [np.percentile(np.log(mi_sub["gdp_pc_ppp"].dropna()), q)
               for q in [10,25,50,75,90]]
    print(f"\n  log(y)   y(PPP)   marginal effect of G*sigma_KIS    std.err   p")
    print("  "+"-"*68)
    for ly in deciles:
        me = b1 + b2*ly + b3*ly**2
        grad = np.array([1, ly, ly**2])
        se = np.sqrt(grad @ v @ grad)
        z = me/se if se > 0 else np.nan
        from scipy.stats import norm
        p = 2*(1 - norm.cdf(abs(z))) if not np.isnan(z) else np.nan
        print(f"  {ly:5.2f}   ${np.exp(ly):>7,.0f}        {me:+9.2f}            "
              f"{se:6.2f}   {p:.3f}")

# -----------------------------------------------------------------------------
# SPEC E — UMIC placebo
# -----------------------------------------------------------------------------
print("\n" + "="*72)
print("SPEC E — UMIC placebo (should be flat if mediation story is right)")
print("="*72)
mE, _ = fit_panel(um_sub, regs_A, outcome="g_outcome")
if mE is not None:
    print_result("UMIC sub-sample, baseline (should be null)", mE, regs_A)

# -----------------------------------------------------------------------------
# SPEC F — Triple interaction: did the income mediation kill the G*sk effect?
# -----------------------------------------------------------------------------
print("\n" + "="*72)
print("SPEC F — Decisive test: G*sk effect AT THE LMIC INCOME LEVEL")
print("        (compares Spec A and Spec C marginal effects at same point)")
print("="*72)
if mA is not None and mC is not None:
    print(f"\nSpec A (LMIC binning):    G*sigma_KIS = {mA.params['G_sk_l']:+8.2f}  "
          f"(p={mA.pvalues['G_sk_l']:.3f})")

    # Mean log GDP in LMIC sub-sample
    ly_lmic = np.log(lm_sub["gdp_pc_ppp"]).mean()
    print(f"\nLMIC mean log(y) = {ly_lmic:.2f}  (y ≈ ${np.exp(ly_lmic):,.0f} PPP)")

    b1 = mC.params.get("G_sk_l", 0)
    b2 = mC.params.get("G_sk_l_x_logy", 0)
    b3 = mC.params.get("G_sk_l_x_logy_sq", 0)
    me_at_lmic = b1 + b2*ly_lmic + b3*ly_lmic**2

    grad = np.array([1, ly_lmic, ly_lmic**2])
    v = mC.cov_params().loc[
        ["G_sk_l","G_sk_l_x_logy","G_sk_l_x_logy_sq"],
        ["G_sk_l","G_sk_l_x_logy","G_sk_l_x_logy_sq"]
    ].values
    se = np.sqrt(grad @ v @ grad)
    print(f"Spec C at that income: marginal effect = {me_at_lmic:+8.2f}  "
          f"(se={se:.2f})")
    if abs(me_at_lmic) < abs(mA.params['G_sk_l'])*0.5:
        print("\n=> Spec C ABSORBS the LMIC negative effect via income mediation.")
        print("   Interpretation: premature deindustrialization is the confound.")
    else:
        print("\n=> Effect SURVIVES income mediation.")
        print("   Interpretation: G*sigma_KIS captures something real beyond income.")

print("\nDone.\n")

Growth-like columns found: ['gdp_pc_growth', 'gdp_pc_growth_raw']
Using growth variable: gdp_pc_growth

SPEC A — Baseline (replicate LMIC result)

LMIC sub-sample, baseline  (N=372)
------------------------------------------------------------------------
                                coef       se       t       p
G_l                           7.3374   2.1433  3.4235  0.0006
sigma_kis                    24.3314  31.2713  0.7781  0.4365
G_sk_l                      -44.5318  29.7270 -1.4980  0.1341
log_gdp_l                   -12.0675   2.8936 -4.1704  0.0000
trade_pct_gdp_l               0.0438   0.0283  1.5483  0.1216
pop_growth_l                  0.2170   0.7500  0.2894  0.7723
inflation_l                  -0.0021   0.0026 -0.8090  0.4185
investment_pct_gdp_l          0.0864   0.0712  1.2134  0.2250
credit_to_private_pct_gdp_l  -0.0112   0.0189 -0.5904  0.5549

SPEC B — + linear income mediation of G*sigma_KIS

LMIC sub-sample, linear mediation  (N=372)
------------------------------

In [ ]:
# ============================================================================
# INFORMALITY CHANNEL TEST — fully self-contained Colab cell
# Pops up upload button, then runs the full analysis
# ============================================================================

import os, glob
import numpy as np
import pandas as pd
import statsmodels.api as sm
from google.colab import files

PANEL_PATH = "final_panel_F_rebuilt.csv"

# ---- UPLOAD STEP -----------------------------------------------------------
print("Click the button below and select your informality CSV")
print("(from https://data.worldbank.org/indicator/SL.ISV.IFRM.ZS)")
print("-" * 60)
uploaded = files.upload()
if not uploaded:
    raise SystemExit("No file uploaded — re-run this cell.")
INFORMAL_PATH = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {INFORMAL_PATH}  ({len(uploaded[INFORMAL_PATH]):,} bytes)")

# ---- LOAD PANEL ------------------------------------------------------------
if not os.path.exists(PANEL_PATH):
    print(f"\n{PANEL_PATH} not found — upload it now:")
    panel_up = files.upload()
df = (pd.read_csv(PANEL_PATH)
        .drop_duplicates(subset=["iso3", "year"])
        .sort_values(["iso3", "year"])
        .reset_index(drop=True))
print(f"\nPanel: {len(df):,} rows, {df['iso3'].nunique()} countries")

# ---- PARSE INFORMALITY FILE ------------------------------------------------
with open(INFORMAL_PATH) as f:
    head = f.read(2000)

if "Country Code" in head or "SL.ISV.IFRM" in head:
    print("Detected: World Bank wide format")
    for skip in [0, 1, 2, 3, 4]:
        try:
            d = pd.read_csv(INFORMAL_PATH, skiprows=skip)
            if "Country Code" in d.columns:
                break
        except Exception:
            continue
    yr_cols = [c for c in d.columns if str(c).isdigit() and 1990 <= int(c) <= 2030]
    info = (d.melt(id_vars=["Country Code"], value_vars=yr_cols,
                   var_name="year", value_name="informal_share")
              .rename(columns={"Country Code": "iso3"})
              .dropna(subset=["informal_share"]))
    info["year"] = info["year"].astype(int)
elif "ref_area" in head or "obs_value" in head or "SDG_0831" in head:
    print("Detected: ILOSTAT long format")
    d = pd.read_csv(INFORMAL_PATH)
    if "sex" in d.columns:
        d = d[d["sex"].str.contains("Total|Sexes", case=False, na=False)]
    sector_col = next((c for c in ["classif1","sector"] if c in d.columns), None)
    if sector_col:
        d = d[d[sector_col].str.contains("Total|Non-agriculture",
                                          case=False, na=False, regex=True)]
    info = (d.rename(columns={"ref_area":"iso3","time":"year",
                              "obs_value":"informal_share"})
              [["iso3","year","informal_share"]].dropna())
    info["year"] = info["year"].astype(int)
else:
    print("Unknown format — attempting generic parse")
    info = pd.read_csv(INFORMAL_PATH)
    rename_map = {}
    for c in info.columns:
        if c.lower() in ("country code","iso3","ref_area","countrycode"):
            rename_map[c] = "iso3"
        elif c.lower() in ("year","time","date"):
            rename_map[c] = "year"
        elif c.lower() in ("obs_value","value","informal","informal_share"):
            rename_map[c] = "informal_share"
    info = (info.rename(columns=rename_map)
                [["iso3","year","informal_share"]].dropna())
    info["year"] = info["year"].astype(int)

print(f"Parsed: {len(info):,} country-year obs, {info['iso3'].nunique()} countries")

# ---- MERGE -----------------------------------------------------------------
df = df.merge(info, on=["iso3","year"], how="left")
n_match = df["informal_share"].notna().sum()
print(f"\nMerged: {n_match:,} country-years matched ({100*n_match/len(df):.1f}%)")
print("Coverage by income group:")
print(df.groupby("income_group")["informal_share"].agg(["count","mean"]).round(2))

# ---- BUILD LAGGED REGRESSORS -----------------------------------------------
df["informal_l"]  = df.groupby("iso3")["informal_share"].shift(1)
df["log_gdp_l"]   = df.groupby("iso3").apply(
                      lambda g: np.log(g["gdp_pc_ppp"]).shift(1)
                    ).reset_index(level=0, drop=True)
df["G_sk_l"]      = df["G_l"] * df["sigma_kis"]
df["G_inf_l"]     = df["G_l"] * df["informal_l"]
df["G_sk_inf_l"]  = df["G_l"] * df["sigma_kis"] * df["informal_l"]

# ---- REGRESSION HELPER -----------------------------------------------------
def fit_fe(data, regs, outcome, entity="iso3"):
    sub = data.dropna(subset=regs+[outcome, entity, "year"]).copy()
    if len(sub) < 30:
        return None
    for c in regs+[outcome]:
        sub[c] = (sub[c] - sub.groupby(entity)[c].transform("mean")
                          - sub.groupby("year")[c].transform("mean")
                          + sub[c].mean())
    X = sm.add_constant(sub[regs])
    return sm.OLS(sub[outcome], X).fit(cov_type="cluster",
                                        cov_kwds={"groups":sub[entity]})

def report(name, m, focus):
    if m is None:
        print(f"\n{name}: too few obs"); return
    print(f"\n{name}  (N={int(m.nobs)})")
    print("-"*68)
    out = pd.DataFrame({"coef":m.params,"se":m.bse,
                        "t":m.tvalues,"p":m.pvalues})
    print(out.loc[[f for f in focus if f in m.params.index]].round(4))

ctrls = [c for c in ["log_gdp_l","trade_pct_gdp_l","pop_growth_l",
                     "inflation_l","investment_pct_gdp_l"] if c in df.columns]

# ---- (a) DIRECT TEST -------------------------------------------------------
print("\n" + "="*68)
print("(a) DIRECT TEST — does AI raise informal employment?")
print("    Informal_t ~ G + sigma_KIS + G*sigma_KIS + controls")
print("="*68)
regs_a = ["G_l","sigma_kis","G_sk_l"] + ctrls
report("All countries", fit_fe(df, regs_a, "informal_share"),
       ["G_l","sigma_kis","G_sk_l"])
report("LMIC only",
       fit_fe(df[df["income_group"]=="LM"], regs_a, "informal_share"),
       ["G_l","sigma_kis","G_sk_l"])
report("UMIC only",
       fit_fe(df[df["income_group"]=="UM"], regs_a, "informal_share"),
       ["G_l","sigma_kis","G_sk_l"])

# ---- (b) MEDIATOR TEST -----------------------------------------------------
print("\n" + "="*68)
print("(b) MEDIATOR TEST — does informality mediate the G*sigma_KIS growth effect?")
print("="*68)
mi = df[df["mi_status"]==1].copy()
regs_b = ["G_l","sigma_kis","G_sk_l","informal_l","G_inf_l","G_sk_inf_l"] + ctrls
report("All MI sample", fit_fe(mi, regs_b, "gdp_pc_growth"),
       ["G_l","sigma_kis","G_sk_l","informal_l","G_inf_l","G_sk_inf_l"])

# ---- (c) ATTENUATION TEST --------------------------------------------------
print("\n" + "="*68)
print("(c) ATTENUATION TEST — does G*sigma_KIS shrink when informality enters?")
print("="*68)
m_base = fit_fe(mi, ["G_l","sigma_kis","G_sk_l"] + ctrls, "gdp_pc_growth")
m_add  = fit_fe(mi, ["G_l","sigma_kis","G_sk_l","informal_l"] + ctrls,
                "gdp_pc_growth")
if m_base is not None and m_add is not None:
    b0 = m_base.params["G_sk_l"]
    b1 = m_add.params["G_sk_l"]
    print(f"\n  Without Informal_l: G*sigma_KIS = {b0:+.2f} "
          f"(p={m_base.pvalues['G_sk_l']:.3f})")
    print(f"  With    Informal_l: G*sigma_KIS = {b1:+.2f} "
          f"(p={m_add.pvalues['G_sk_l']:.3f})")
    if b0 != 0:
        pct = 100*(b1-b0)/b0
        print(f"  Change: {pct:+.1f}%")
        if abs(b1) < 0.5*abs(b0):
            print("  => Informality ABSORBS the effect: plausible mediator.")
        else:
            print("  => G*sigma_KIS survives: informality is NOT the mediator.")

# ---- SAVE ------------------------------------------------------------------
df.to_csv("final_panel_with_informality.csv", index=False)
print("\nSaved: final_panel_with_informality.csv")

Click the button below and select your informality CSV
(from https://data.worldbank.org/indicator/SL.ISV.IFRM.ZS)
------------------------------------------------------------
